In [1]:
import os.path as osp
import sys
sys.path.append('../')
import torch
from torch.utils.data import DataLoader

import numpy as np
from sklearn import metrics
from tqdm.notebook import tqdm
import types
import copy
import vector

import matplotlib
import matplotlib.pyplot as plt

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

import utils

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE = 'cpu'
print(DEVICE)

cuda:0


/home/mdrnevich/dev/QuasiML_paper/QuasiML/SMEFT/../utils/__init__.py:5: UserWarning: The recommended fonts to use plothist were not found. You can install them by typing 'install_latin_modern_fonts' in your terminal. 

  from . import plotting


In [7]:
DATA_NUM = ""
TEST_NUM = ""

In [8]:
submodel_names = np.array(["Pos", "Neg", "Pos", "Neg"])
combos = [np.array((0,2)), #++
          np.array((0,3))] #+-

NUM_MODELS = 2

# First load the models from the previous paper

In [9]:
batch_sizes = [int(2**6), int(2**8)]
submodel_paths = [osp.join("models/classifier{}_subdensity_{}_batch{}.zip".format(2, ''.join(submodel_names[combos[i]]), batch_sizes[i])) for i in range(NUM_MODELS)]

In [10]:
## Load the coefficient initialization

source_file = "./data/SMEFT_SM_combined_tuple".format(DATA_NUM)
target_file = "./data/SMEFT_EFT_combined_tuple".format(DATA_NUM)

source_positive_file = source_file + "_positives"
source_negative_file = source_file + "_negatives"
target_positive_file = target_file + "_positives"
target_negative_file = target_file + "_negatives"

files = [source_positive_file, source_negative_file, target_positive_file, target_negative_file]

sum_weights = []
for f in files:
    sum_weights.append(np.load(f + "_train.npy")[:,-1].sum())

sum_weights = np.array(sum_weights)
coefficient_init = [sum_weights[0] / sum_weights[:2].sum(), sum_weights[2] / sum_weights[2:].sum()]

## Now create the model

batch_size = int(2**8)
modpath = osp.join("models/classifier2{}_mixture_batch{}.zip".format(2, batch_size))

old_SMM_model = utils.models.SingleMixtureClassifier(submodel_paths, fine_tune=False).to(DEVICE)
X_scaler_old_SMM, weight_norm_old_SMM = utils.preprocessing.load_scaling(modpath)
old_SMM_model.initialize(coefficient_init)
old_SMM_model.coefficients

(1.0, 1.267315149307251)

In [11]:
batch_size = int(2**8)
modpath = osp.join("models/classifier2{}_mixture_batch{}.zip".format(2, batch_size))

old_SMMc_model = utils.models.SingleMixtureClassifier(submodel_paths, fine_tune=False).to(DEVICE)
X_scaler_old_SMMc, weight_norm_old_SMMc = utils.preprocessing.load_scaling(modpath)
old_SMMc_model.initialize((1.0, 1.2691593170166016))
old_SMMc_model.coefficients

(1.0, 1.2691593170166016)

In [12]:
batch_size = int(2**9)
modpath2 = osp.join("models/classifier2{}_mixture_batch{}_fine_tune.zip".format(2, batch_size))

old_SMMr_model = utils.models.SingleMixtureClassifier(submodel_paths, fine_tune=True).to(DEVICE)
old_SMMr_model = utils.models.load_model_state(modpath2, old_SMMr_model, device=DEVICE).to(DEVICE)

X_scaler_old_SMMr, weight_norm_old_SMMr = utils.preprocessing.load_scaling(modpath2)
old_SMMr_model.coefficients

(1.0, 1.2669531106948853)

# Now load the new models

## Load the SMM model

In [13]:
batch_size = int(2**6)
TEST_NUM = ""
modpath = osp.join("models/classifier{}_SMM.zip".format(TEST_NUM, batch_size))
SMM_model = utils.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_SMM, weight_norm_SMM = utils.preprocessing.load_scaling(modpath)
SMM_model.coefficients

(1.0, 1.267315149307251)

In [14]:
print(SMM_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_c$ model

In [34]:
batch_size = int(2**6)
TEST_NUM = ""
modpath = osp.join("models/classifier{}_SMMc_qdre_batch{}.zip".format(TEST_NUM, batch_size))
SMMc_model = utils.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_SMMc, weight_norm_SMMc = utils.preprocessing.load_scaling(modpath)
SMMc_model.coefficients

(1.0, 1.2870757579803467)

In [35]:
print(SMMc_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_r$ model

In [38]:
batch_size = int(2**6)
TEST_NUM = "" #"2"
modpath = osp.join("models/classifier{}_SMMr_qdre_batch{}.zip".format(TEST_NUM, batch_size))
SMMr_model = utils.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_SMMr, weight_norm_SMMr = utils.preprocessing.load_scaling(modpath)
SMMr_model.coefficients

(1.0, 1.2711114883422852)

In [39]:
print(SMMr_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the new MLP classifier

In [40]:
NEW_LOSS = "qdre" # "qdre"
TEST_NUM = "3" #"3"
batch_size = int(2**8)
modpath = osp.join("models/classifier{}_{}_batch{}.zip".format(TEST_NUM, NEW_LOSS, batch_size))
model_new = utils.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_new, weight_norm_new = utils.preprocessing.load_scaling(modpath)

In [41]:
print(model_new)

Classifier(
  (model): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## Do the evaluation plots

In [42]:
@torch.no_grad()
def get_r_hats(model, loader, X_scaler=None, weight_norm=1, mix=False, leave=False, loss="bce", t0=None, t1=None):
    if type(model) is not types.FunctionType:
        model.eval()
        if mix is True:
            loader.collate_fn = lambda batch: utils.preprocessing.prep_inputs_for_training_mix(batch, weight_norm=weight_norm)
        else:
            loader.collate_fn = lambda batch: utils.preprocessing.prep_inputs_for_training(batch, X_scaler, weight_norm=weight_norm)
    else:
        loader.collate_fn = lambda batch: utils.preprocessing.prep_inputs_for_density(batch, weight_norm=weight_norm)

    r_hat_list = []
    t = tqdm(enumerate(loader), total=len(loader), leave=leave)
    for i, batch in t:
        if type(model) is not types.FunctionType:
            x = batch[0].to(DEVICE)
        else:
            x = batch[0].to('cpu')
        
        if mix is True:
            r_hat = model.ratio(x)
        else:
            batch_output = model(x)
            if loss in ["bce", "mse"]:
                r_hat = batch_output / (1 - batch_output)
            elif loss == "pare":
                r_hat = - t0*(1- t0*batch_output)/(t1*(1 - t1*batch_output))
            elif loss == 'qdre':
                r_hat = (1-2*batch_output) / (batch_output - batch_output**2)
            elif loss == 'new':
                r_hat = torch.log(1-batch_output) - torch.log(batch_output)
            elif loss == 'new2':
                r_hat = -torch.tan(torch.pi*(batch_output - 0.5))
        r_hat_list.append(r_hat)
        t.refresh()  # to show immediately the update

    return torch.cat(r_hat_list).cpu().numpy().flatten()

In [43]:
MODEL_LIST = [model_new, old_SMM_model, old_SMMc_model, old_SMMr_model, SMMc_model, SMMr_model]
MODEL_NAMES = ["new", "SMM", "old_SMMc", "old_SMMr", "SMMc", "SMMr"]
MODEL_SETTINGS = [
    {"X_scaler": X_scaler_new, "weight_norm": weight_norm_new, "loss": NEW_LOSS, "leave": False},
    {"X_scaler": None, "weight_norm": weight_norm_old_SMM, "mix":True, "leave": False},
    {"X_scaler": None, "weight_norm": weight_norm_old_SMMc, "mix":True, "leave": False},
    {"X_scaler": None, "weight_norm": weight_norm_old_SMMr, "mix":True, "leave": False},
    {"X_scaler": None, "weight_norm": weight_norm_SMMc, "mix":True, "leave": False},
    {"X_scaler": None, "weight_norm": weight_norm_SMMr, "mix":True, "leave": False}
]

In [44]:
SAVE = True

if SAVE is True:
    for dataset in ["train", "val", "test"]:
        base_dataset = utils.preprocessing.Dataset(source_file + "_" + dataset + ".npy", 0)
        base_dataset.process()
        nominal_loader = DataLoader(utils.preprocessing.CombinedDataset(base_dataset), batch_size=int(2**10), shuffle=False)
        
        for i in range(len(MODEL_LIST)):
            r_hats = get_r_hats(
                MODEL_LIST[i],
                nominal_loader,
                **MODEL_SETTINGS[i]
            )
            print(r_hats.shape)

            arr = np.load(source_file + "_" + dataset + ".npy")
            arr[:, -1] = abs(r_hats*arr[:, -1])
            np.save("{}_discriminant_{}_{}.npy".format(source_file, MODEL_NAMES[i], dataset), arr)

del arr

  0%|          | 0/759 [00:00<?, ?it/s]

(776369,)


  0%|          | 0/759 [00:00<?, ?it/s]

(776369,)


  0%|          | 0/759 [00:00<?, ?it/s]

...

(238883,)


  0%|          | 0/234 [00:00<?, ?it/s]

(238883,)


  0%|          | 0/234 [00:00<?, ?it/s]

(238883,)
